# Week 8 (W8-2) - Supplier risk: a transparent score and how firm its ranking is
**Owner:** Abdibasid Mohamed Ali (Data Scientist)

**Question.** How exposed are we to each of the 8 suppliers, and is the resulting risk ranking firm enough to act on?

**Basis.** The unit is the **purchase order** (24,000 POs; 21,630 received, 2,370 cancelled). The risk score has three equal pillars, each scaled 0 to 1 across the 8 suppliers (0 = lowest risk, 1 = highest) and averaged, then multiplied by 100:
1. **Reliability** = mean of the late-delivery rate (received POs not on or before the expected date) and the cancellation rate (all POs).
2. **Concentration** = the supplier's share of received purchase spend (before GST).
3. **Lead-time spread** = standard deviation of actual lead time (received date minus order date) within the supplier, in days.

No weight is fitted; the score is a plain, auditable sum. **Firmness** is tested three ways: (a) bootstrap of the POs (5,000 resamples, seed 42) giving a rank interval for every supplier and the probability that any pair is in the stated order; (b) a split-half test, where the same score is built on two random halves of the POs and the two rankings are compared, against halves whose supplier labels were shuffled; (c) a sensitivity run over alternative weights and scalings.

**Also in this notebook.** The Week 7 forecast benchmark is rechecked on the merged data (section 7).

**Limits.** No price-quote, quality or defect fields exist and there is no cost of a late delivery, so risk here means timing, cancellation, spend concentration and lead-time spread only. Purchase spend is about 14 times net sales (Q10 unresolved), so **money figures are descriptive and carry the label `PROCUREMENT SCALE - purchase spend is about 14x sales revenue (Q10 unresolved); money figures are descriptive`**. Rates, shares and lead times do not depend on that scale. A min-max score always spreads 8 suppliers from 0 to 100, so the score size alone says nothing about whether the suppliers really differ; that is what the firmness tests are for.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
SEED=42; NB=5000
PROC_LABEL="PROCUREMENT SCALE - purchase spend is about 14x sales revenue (Q10 unresolved); money figures are descriptive"
def find_dir(env_var, marker, rels):
    # 1) environment variable, 2) known layouts from here upward, 3) bounded search of the parent folders
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    bases = [Path.cwd(), *Path.cwd().parents]
    for base in bases:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    skip = {".git", "node_modules", "__pycache__", ".ipynb_checkpoints", ".venv", "venv", "site-packages"}
    for base in bases[:4]:
        found = []
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in skip]
            if len(Path(root).parts) - len(base.parts) >= 7:
                dirs[:] = []
            if marker in files:
                found.append(Path(root))
        if found:
            found.sort(key=lambda p: (p.name != Path(rels[0]).name, len(p.parts), str(p)))
            return found[0]
    return None
MERGED = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged", "week-02/reports/merged", "reports/merged"])
assert MERGED is not None, "Could not find the merged folder. Run from inside the repository, or set CADETX_MERGED to the folder."
print("merged data folder found")
OUT=Path("w82_outputs"); OUT.mkdir(exist_ok=True); CH=Path("w82_charts"); CH.mkdir(exist_ok=True)
rng=np.random.default_rng(SEED)
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    for s in ["left","bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True,color=GRID,lw=.8); ax.set_axisbelow(True)
def minmax(v):
    v=np.asarray(v,float); r=v.max(axis=-1,keepdims=True)-v.min(axis=-1,keepdims=True)
    return np.where(r>0,(v-v.min(axis=-1,keepdims=True))/np.where(r>0,r,1),0.0)

merged data folder found


## 1. Load and reconcile

In [2]:
L=pd.read_csv(MERGED/"purchase_orders_full.csv",low_memory=False,parse_dates=["order_date","expected_delivery_date","received_date"])
check("PO lines",155495,len(L),len(L)==155495)
H=L.drop_duplicates("po_id").copy().reset_index(drop=True)
check("purchase orders",24000,len(H),len(H)==24000)
var=int((L.groupby("po_id")[["supplier_id","total_cost","po_status","order_date","expected_delivery_date"]].nunique()>1).sum().sum()); check("header fields are constant within each PO",0,var,var==0)
S=np.array(sorted(H.supplier_id.unique())); check("eight suppliers",8,len(S),len(S)==8)
NAME=H.drop_duplicates("supplier_id").set_index("supplier_id").supplier_name
REGION=H.drop_duplicates("supplier_id").set_index("supplier_id").supplier_region
rec=(H.po_status=="Received").values; can=(H.po_status=="Cancelled").values
check("received plus cancelled equals all POs",24000,int(rec.sum()+can.sum()),int(rec.sum()+can.sum())==24000)
check("received POs",21630,int(rec.sum()),int(rec.sum())==21630)
check("received spend before GST (INR B)",312.03,round(H.total_cost[rec].sum()/1e9,2),abs(H.total_cost[rec].sum()/1e9-312.03)<0.005)
d1=int(((H.received_date-H.expected_delivery_date).dt.days[rec]!=H.delivery_delay_days[rec]).sum()); check("delay = received date - expected date on every received PO",0,d1,d1==0)
d2=int(((H.received_date-H.order_date).dt.days[rec]!=H.lead_time_actual_days[rec]).sum()); check("lead time = received date - order date on every received PO",0,d2,d2==0)
sid=pd.Categorical(H.supplier_id,categories=S).codes
late=((H.delivery_delay_days>0)&rec).values.astype(float)
ontime=((H.delivery_delay_days<=0)&rec).values.astype(float)
x=np.where(rec,H.lead_time_actual_days.fillna(0).values,0.0)
cost=np.where(rec,H.total_cost.values,0.0)
print("on-time overall %.1f%% of received POs"%(ontime.sum()/rec.sum()*100))

on-time overall 84.9% of received POs


## 2. The risk score
Each pillar is computed per supplier from the POs, then min-max scaled across the 8 suppliers. The raw component values are kept next to the score so every number can be traced.

In [3]:
def components(w):
    """w = weight of each PO (1 for the data, bootstrap counts for a resample). Returns 8 x 4 raw components."""
    k=len(S); f=lambda v: np.bincount(sid,weights=w*v,minlength=k)
    n=f(np.ones(len(H))); nc=f(can.astype(float)); nr=f(rec.astype(float)); nl=f(late); no=f(ontime)
    sp=f(cost); s1=f(x); s2=f(x*x)
    late_rate=nl/nr*100; cancel_rate=nc/n*100; share=sp/sp.sum()*100
    sd=np.sqrt(np.maximum((s2-s1**2/nr)/(nr-1),0))
    return np.column_stack([late_rate,cancel_rate,share,sd])
def score_from(C,wts=(1/3,1/3,1/3),how="minmax"):
    C=np.asarray(C,float)
    if how=="minmax": N=np.stack([minmax(C[...,j]) for j in range(4)],axis=-1)
    elif how=="z": N=(C-C.mean(axis=-2,keepdims=True))/C.std(axis=-2,keepdims=True)
    elif how=="rank": N=np.apply_along_axis(lambda v: stats.rankdata(v),-2,C)
    rel=(N[...,0]+N[...,1])/2
    s=wts[0]*rel+wts[1]*N[...,2]+wts[2]*N[...,3]
    return s*100 if how=="minmax" else s
C0=components(np.ones(len(H))); sc0=score_from(C0)
rank0=stats.rankdata(-sc0,method="min").astype(int)  # 1 = riskiest
SC=pd.DataFrame(dict(supplier_id=S,supplier_name=NAME[S].values,region=REGION[S].values,
    late_rate_pct=C0[:,0],cancel_rate_pct=C0[:,1],spend_share_pct=C0[:,2],lead_time_sd_days=C0[:,3],
    reliability_pillar=(minmax(C0[:,0])+minmax(C0[:,1]))/2,concentration_pillar=minmax(C0[:,2]),spread_pillar=minmax(C0[:,3]),
    risk_score=sc0,risk_rank=rank0,label=PROC_LABEL)).sort_values("risk_rank")
SC.round(4).to_csv(OUT/"w82_risk_scorecard.csv",index=False)
print(SC[["supplier_id","late_rate_pct","cancel_rate_pct","spend_share_pct","lead_time_sd_days","risk_score","risk_rank"]].round(2).to_string(index=False))
# independent recomputation with plain filters for three suppliers
for j in [0,3,6]:
    h=H[H.supplier_id==S[j]]; r=h[h.po_status=="Received"]
    exp=[(r.delivery_delay_days>0).mean()*100,(h.po_status=="Cancelled").mean()*100,r.total_cost.sum()/H[rec].total_cost.sum()*100,r.lead_time_actual_days.std()]
    check(f"{S[j]} four components recomputed with plain filters",np.round(exp,4).tolist(),np.round(C0[j],4).tolist(),np.allclose(exp,C0[j],atol=1e-8))
check("spend shares add to 100",100.0,round(C0[:,2].sum(),6),abs(C0[:,2].sum()-100)<1e-6)
_ok=all(abs(minmax(C0[:,j]).min())<1e-12 and abs(minmax(C0[:,j]).max()-1)<1e-12 for j in range(4)); check("each of the four components scales to exactly 0 and 1 across suppliers",True,_ok,_ok)
check("risk score recomputed by hand for the riskiest supplier",round(SC.risk_score.iloc[0],6),round(100/3*((minmax(C0[:,0])+minmax(C0[:,1]))/2+minmax(C0[:,2])+minmax(C0[:,3]))[SC.index[0]],6),abs(SC.risk_score.iloc[0]-100/3*((minmax(C0[:,0])+minmax(C0[:,1]))/2+minmax(C0[:,2])+minmax(C0[:,3]))[SC.index[0]])<1e-9)
HHI=float(((C0[:,2]/100)**2).sum()); check("spend HHI is at least 1/8",True,round(HHI,4),HHI>=0.125-1e-12)
print("spend HHI %.4f (minimum possible with 8 suppliers 0.1250)"%HHI)

supplier_id  late_rate_pct  cancel_rate_pct  spend_share_pct  lead_time_sd_days  risk_score  risk_rank
    SUP0003          14.98            10.22            12.56               1.78       72.78          1
    SUP0008          15.80             9.57            12.58               1.72       58.42          2
    SUP0007          15.33             9.94            12.51               1.73       56.48          3
    SUP0001          15.08             9.88            12.64               1.72       55.56          4
    SUP0002          15.03            10.28            12.09               1.72       40.53          5
    SUP0006          14.79             9.41            12.53               1.72       38.06          6
    SUP0005          14.67             9.53            12.90               1.65       35.50          7
    SUP0004          15.17            10.18            12.19               1.69       35.40          8
spend HHI 0.1250 (minimum possible with 8 suppliers 0.1250)


## 3. How firm is the ranking? Bootstrap of the purchase orders
Each of the 5,000 resamples draws 24,000 POs with replacement (so PO counts and spend per supplier move as well), recomputes the four components and the score, and ranks the suppliers (1 = riskiest). We report each supplier's rank interval, the chance it is the riskiest or in the top three, and for every pair the share of resamples in which the base order holds.

In [4]:
k=len(S); N=len(H)
cols=[np.eye(k)[sid]*v[:,None] for v in [np.ones(N),can.astype(float),rec.astype(float),late,ontime,cost,x,x*x]]
M=np.hstack(cols)   # N x (8*8)
BS=np.empty((NB,k,4)); done=0
rng_b=np.random.default_rng(SEED)
while done<NB:
    m=min(250,NB-done)
    W=rng_b.multinomial(N,np.full(N,1.0/N),size=m).astype(float)   # bootstrap counts per PO
    T=(W@M).reshape(m,8,k)
    n,nc,nr,nl,no,sp,s1,s2=[T[:,i,:] for i in range(8)]
    sd=np.sqrt(np.maximum((s2-s1**2/nr)/(nr-1),0))
    BS[done:done+m]=np.stack([nl/nr*100,nc/n*100,sp/sp.sum(axis=1,keepdims=True)*100,sd],axis=-1); done+=m
check("5,000 bootstrap resamples",5000,len(BS),len(BS)==5000)
chk=components(np.bincount(rng_b.integers(0,N,N),minlength=N).astype(float)); check("bootstrap components have the same shape as the data components",C0.shape,chk.shape,C0.shape==chk.shape)
SB=score_from(BS)                                  # NB x 8
RB=np.apply_along_axis(lambda v: stats.rankdata(-v,method="average"),1,SB)   # 1 = riskiest
pair=np.array([[ (SB[:,i]>SB[:,j]).mean() for j in range(k)] for i in range(k)])
rk=pd.DataFrame(dict(supplier_id=S,supplier_name=NAME[S].values,base_score=sc0,base_rank=rank0,
    rank_lo=np.percentile(RB,2.5,axis=0),rank_median=np.median(RB,axis=0),rank_hi=np.percentile(RB,97.5,axis=0),
    p_riskiest=(RB<=1.0).mean(axis=0),p_top3=(RB<=3.0).mean(axis=0),
    score_lo=np.percentile(SB,2.5,axis=0),score_hi=np.percentile(SB,97.5,axis=0))).sort_values("base_rank")
rk["rank_interval_width"]=rk.rank_hi-rk.rank_lo; rk["label"]=PROC_LABEL
rk.round(4).to_csv(OUT/"w82_rank_stability.csv",index=False)
PM=pd.DataFrame(pair,index=S,columns=S).round(4); PM.to_csv(OUT/"w82_pairwise_order_probability.csv")
print(rk[["supplier_id","base_score","base_rank","rank_lo","rank_median","rank_hi","p_riskiest","p_top3"]].round(3).to_string(index=False))
# a pair is "firm" when the base order holds in at least 95% of resamples
firm=0; tot=0
for i in range(k):
    for j in range(i+1,k):
        p=max(pair[i,j],pair[j,i]); tot+=1; firm+= p>=0.95
print("pairs whose order holds in at least 95%% of resamples: %d of %d"%(firm,tot))
check("pairwise probabilities of i riskier than j and j riskier than i add to 1 (no ties)",28,int(sum(abs(pair[i,j]+pair[j,i]-1)<1e-9 for i in range(k) for j in range(i+1,k))),sum(abs(pair[i,j]+pair[j,i]-1)<1e-9 for i in range(k) for j in range(i+1,k))==28)
check("probabilities of being riskiest add to at least 1 (ties share the top rank)",True,round(float(rk.p_riskiest.sum()),6),rk.p_riskiest.sum()>=1-1e-9)
check("each supplier's rank interval contains its median",8,int(((rk.rank_lo<=rk.rank_median)&(rk.rank_median<=rk.rank_hi)).sum()),((rk.rank_lo<=rk.rank_median)&(rk.rank_median<=rk.rank_hi)).all())
rho_b=np.array([stats.spearmanr(-RB[b],sc0).statistic for b in range(NB)])
print("mean Spearman between each resample's ranking and the base ranking: %.2f"%rho_b.mean())
BASE=pd.DataFrame(dict(statistic=["pairs firm at 95%","pairs total","mean Spearman, resample vs base ranking","share of resamples with same riskiest supplier as base",
    "median width of 95% rank interval (ranks)"],value=[firm,tot,rho_b.mean(),float((RB[:,rank0.argmin()]<=1.0).mean()),float(rk.rank_interval_width.median())]))
BASE.round(4).to_csv(OUT/"w82_ranking_firmness.csv",index=False); print(BASE.round(3).to_string(index=False))

supplier_id  base_score  base_rank  rank_lo  rank_median  rank_hi  p_riskiest  p_top3
    SUP0003      72.785          1    1.000          2.0      7.0       0.415   0.778
    SUP0008      58.423          2    1.000          3.0      8.0       0.153   0.536
    SUP0007      56.481          3    1.000          4.0      8.0       0.142   0.471
    SUP0001      55.560          4    1.000          4.0      8.0       0.147   0.479
    SUP0002      40.527          5    1.000          6.0      8.0       0.038   0.204
    SUP0006      38.060          6    1.000          5.0      8.0       0.058   0.251
    SUP0005      35.499          7    1.975          6.0      8.0       0.025   0.158
    SUP0004      35.400          8    2.000          7.0      8.0       0.022   0.124
pairs whose order holds in at least 95% of resamples: 0 of 28
mean Spearman between each resample's ranking and the base ranking: 0.48
                                             statistic  value
                             

## 4. Is the ranking more than a label? Split-half test against shuffled labels
If suppliers really differ in risk, a ranking built on one random half of the POs should agree with one built on the other half. We compare the agreement (Spearman) across 5,000 random splits with the agreement when supplier labels are shuffled across POs (no real supplier differences by construction). Agreement near the shuffled value means the ranking is noise.

Reading the numbers: the two halves are disjoint pieces of one fixed dataset, so each supplier's deviation from its full-data value is mirrored in the other half. When the real differences between suppliers are small compared with that sampling noise, the agreement between halves is **negative**, not zero; the shuffled-label run has the same mirroring and is the fair yardstick. Evidence that the ranking is real would be agreement clearly above the shuffled value.

In [5]:
rng_s=np.random.default_rng(SEED)
def split_rho(labels):
    perm=rng_s.permutation(N); a=np.zeros(N,bool); a[perm[:N//2]]=True
    out=[]
    for m in (a,~a):
        f=lambda v: np.bincount(labels[m],weights=v[m],minlength=k)
        n=f(np.ones(N)); nc=f(can.astype(float)); nr=f(rec.astype(float)); nl=f(late); sp=f(cost); s1=f(x); s2=f(x*x)
        sd=np.sqrt(np.maximum((s2-s1**2/nr)/(nr-1),0))
        out.append(score_from(np.column_stack([nl/nr*100,nc/n*100,sp/sp.sum()*100,sd])))
    return stats.spearmanr(out[0],out[1]).statistic
real=np.array([split_rho(sid) for _ in range(NB)])
null=np.array([split_rho(rng_s.permutation(sid)) for _ in range(NB)])
u=stats.mannwhitneyu(real,null,alternative="greater"); u2=stats.mannwhitneyu(real,null,alternative="less")
auc=u.statistic/(len(real)*len(null))
SH=pd.DataFrame([dict(version="true supplier labels",mean_rho=real.mean(),sd_rho=real.std(),p5=np.percentile(real,5),p95=np.percentile(real,95)),
                 dict(version="shuffled supplier labels (no real differences)",mean_rho=null.mean(),sd_rho=null.std(),p5=np.percentile(null,5),p95=np.percentile(null,95))])
SH["auc_true_beats_shuffled"]=auc; SH["mannwhitney_p_true_better"]=u.pvalue; SH["mannwhitney_p_true_worse"]=u2.pvalue; SH["splits"]=NB; SH.round(4).to_csv(OUT/"w82_split_half.csv",index=False)
print(SH.round(3).to_string(index=False)); print("AUC %.3f (0.5 = same as shuffled); p that true-label halves agree BETTER than shuffled = %.3f; WORSE = %.4f"%(auc,u.pvalue,u2.pvalue))
check("5,000 true-label and 5,000 shuffled-label splits",10000,len(real)+len(null),len(real)+len(null)==10000)
check("all split-half correlations are valid (between -1 and 1)",True,True,bool(np.all(np.abs(real)<=1+1e-9) and np.all(np.abs(null)<=1+1e-9)))

                                       version  mean_rho  sd_rho     p5   p95  auc_true_beats_shuffled  mannwhitney_p_true_better  mannwhitney_p_true_worse  splits
                          true supplier labels    -0.312   0.303 -0.738 0.238                     0.28                        1.0                       0.0    5000
shuffled supplier labels (no real differences)    -0.028   0.378 -0.667 0.595                     0.28                        1.0                       0.0    5000
AUC 0.280 (0.5 = same as shuffled); p that true-label halves agree BETTER than shuffled = 1.000; WORSE = 0.0000


## 5. Sensitivity to weights and scaling
The ranking is rebuilt under alternative weightings of the three pillars and under z-score and rank scaling, and each is compared with the base ranking.

In [6]:
ALT=[("base: equal pillars, min-max",(1/3,1/3,1/3),"minmax"),("reliability-heavy 50/25/25",(.5,.25,.25),"minmax"),("concentration-heavy 25/50/25",(.25,.5,.25),"minmax"),
     ("spread-heavy 25/25/50",(.25,.25,.5),"minmax"),("equal pillars, z-score",(1/3,1/3,1/3),"z"),("equal pillars, rank scaling",(1/3,1/3,1/3),"rank")]
rows=[]; R_alt={}
for nm,w,h in ALT:
    s=score_from(C0,w,h); r=stats.rankdata(-s,method="average"); R_alt[nm]=r
    rows.append(dict(variant=nm,riskiest=S[np.argmax(s)],least_risky=S[np.argmin(s)],spearman_vs_base=stats.spearmanr(s,sc0).statistic,
        top3=", ".join(S[np.argsort(-s)[:3]])))
SA=pd.DataFrame(rows); SA.round(4).to_csv(OUT/"w82_sensitivity.csv",index=False); print(SA.round(3).to_string(index=False))
n_same=int((SA.riskiest==SA.riskiest.iloc[0]).sum())
print("variants agreeing with the base on the riskiest supplier: %d of %d"%(n_same,len(SA)))
check("base variant reproduces the base ranking (Spearman 1)",1.0,round(float(SA.spearman_vs_base.iloc[0]),6),abs(SA.spearman_vs_base.iloc[0]-1)<1e-9)
check("six scoring variants compared",6,len(SA),len(SA)==6)

                     variant riskiest least_risky  spearman_vs_base                      top3
base: equal pillars, min-max  SUP0003     SUP0004             1.000 SUP0003, SUP0008, SUP0007
  reliability-heavy 50/25/25  SUP0003     SUP0005             0.929 SUP0003, SUP0008, SUP0007
concentration-heavy 25/50/25  SUP0003     SUP0004             0.881 SUP0003, SUP0008, SUP0001
       spread-heavy 25/25/50  SUP0003     SUP0005             0.976 SUP0003, SUP0008, SUP0007
      equal pillars, z-score  SUP0003     SUP0004             0.976 SUP0003, SUP0008, SUP0007
 equal pillars, rank scaling  SUP0003     SUP0006             0.929 SUP0003, SUP0008, SUP0007
variants agreeing with the base on the riskiest supplier: 6 of 6


## 6. Exposure context: spend, critical products, geography and lead time
These columns describe how exposed purchasing is to each supplier; they are not part of the score. Money shares are descriptive (procurement scale).

In [7]:
Lr=L.merge(H[["po_id","po_status"]].rename(columns={"po_status":"st"}),on="po_id"); Lr=Lr[Lr.po_status=="Received"].copy()
Lr["line_cost"]=Lr.line_total
tot_all=Lr.line_cost.sum(); hi=Lr[Lr.criticality_level=="High"]
ex=pd.DataFrame(dict(supplier_id=S))
ex["supplier_name"]=NAME[S].values; ex["region"]=REGION[S].values
ex["spend_share_pct"]=[Lr[Lr.supplier_id==s].line_cost.sum()/tot_all*100 for s in S]
ex["high_criticality_spend_share_of_supplier_pct"]=[hi[hi.supplier_id==s].line_cost.sum()/Lr[Lr.supplier_id==s].line_cost.sum()*100 for s in S]
ex["share_of_all_high_criticality_spend_pct"]=[hi[hi.supplier_id==s].line_cost.sum()/hi.line_cost.sum()*100 for s in S]
ps=Lr.groupby(["product_id","supplier_id"]).line_cost.sum().unstack().reindex(columns=S)
big=ps.idxmax(axis=1).value_counts()
ex["products_where_largest_source"]=[int(big.get(s,0)) for s in S]
ex["contract_lead_days"]=[H[H.supplier_id==s].supplier_lead_days.iloc[0] for s in S]
ex["mean_actual_lead_days"]=[H[(H.supplier_id==s)&rec].lead_time_actual_days.mean() for s in S]
ex["label"]=PROC_LABEL; ex.round(3).to_csv(OUT/"w82_exposure_context.csv",index=False); print(ex.drop(columns=["label","supplier_name"]).round(2).to_string(index=False))
reg=ex.groupby("region").spend_share_pct.sum().sort_values(ascending=False); reg.round(3).to_csv(OUT/"w82_region_exposure.csv"); print(reg.round(1).to_string())
check("line-level spend equals PO-level spend on received POs (INR B)",round(H.total_cost[rec].sum()/1e9,2),round(tot_all/1e9,2),abs(H.total_cost[rec].sum()-tot_all)/H.total_cost[rec].sum()<1e-6)
check("products where a supplier is the largest source add to 30",30,int(ex.products_where_largest_source.sum()),int(ex.products_where_largest_source.sum())==30)
check("region shares add to 100",100.0,round(float(reg.sum()),6),abs(reg.sum()-100)<1e-6)
check("every supplier is in China",True,True,set(H.supplier_region.unique())<= {"East China","South China","North China"})

supplier_id      region  spend_share_pct  high_criticality_spend_share_of_supplier_pct  share_of_all_high_criticality_spend_pct  products_where_largest_source  contract_lead_days  mean_actual_lead_days
    SUP0001 South China            12.64                                         93.95                                    12.66                              2                  19                  19.50
    SUP0002 South China            12.09                                         93.63                                    12.06                              4                  10                  10.52
    SUP0003  East China            12.56                                         94.06                                    12.59                              2                  16                  16.53
    SUP0004  East China            12.19                                         93.80                                    12.19                              1                  18              

## 7. Week 7 forecast benchmark: does it still hold on the merged data?
Monthly net revenue (Delivered orders, `line_total` before GST) is rebuilt from the merged sales file. Three simple benchmarks forecast each month of 2023 and 2024 one step ahead using only earlier months: **same month last year** (seasonal naive), **mean of the previous 12 months** and **last month** (naive). Accuracy is the mean absolute percentage error (MAPE), with a bootstrap interval over the 24 test months (5,000 resamples, seed 42).

The Week 7 benchmark figures were not part of this upload, so this section re-derives the benchmark ranking from the data and checks that the inputs reconcile to the Week 7 basis (18,033 delivered orders, 22.96 INR B); it does not claim a match to Week 7 numbers. The check to make is whether the ordering of benchmarks matches the Week 7 write-up.

In [8]:
SO=pd.read_csv(MERGED/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date"],usecols=["so_id","order_date","order_status","line_total"])
dl=SO[SO.order_status=="Delivered"]
check("delivered orders (Week 7 basis)",18033,dl.so_id.nunique(),dl.so_id.nunique()==18033)
check("delivered net revenue before GST (INR B, Week 7 basis)",22.96,round(dl.line_total.sum()/1e9,2),abs(dl.line_total.sum()/1e9-22.96)<0.005)
ms=dl.groupby(dl.order_date.dt.to_period("M")).line_total.sum(); ms.index=ms.index.to_timestamp()
check("72 months from 2019-01 to 2024-12",72,len(ms),len(ms)==72 and ms.index.min()==pd.Timestamp("2019-01-01") and ms.index.max()==pd.Timestamp("2024-12-01"))
y=ms.values; idx=list(range(len(y))); test=[i for i in idx if ms.index[i].year>=2023]
f_sn=np.array([y[i-12] for i in test]); f_mean=np.array([y[i-12:i].mean() for i in test]); f_naive=np.array([y[i-1] for i in test]); act=y[test]
def mape(f): return np.abs(f-act)/act*100
rows=[]
rb=np.random.default_rng(SEED)
for nm,f in [("same month last year",f_sn),("mean of previous 12 months",f_mean),("last month (naive)",f_naive)]:
    e=mape(f); bs=np.array([e[rb.integers(0,len(e),len(e))].mean() for _ in range(NB)])
    rows.append(dict(benchmark=nm,test_months=len(e),mape_pct=e.mean(),mape_lo=np.percentile(bs,2.5),mape_hi=np.percentile(bs,97.5)))
FB=pd.DataFrame(rows); FB.round(3).to_csv(OUT/"w82_forecast_benchmark_recheck.csv",index=False); print(FB.round(2).to_string(index=False))
best=FB.loc[FB.mape_pct.idxmin(),"benchmark"]; print("lowest MAPE:",best)
# plain-loop recomputation of one benchmark
e2=[abs(y[i-12:i].sum()/12-y[i])/y[i]*100 for i in test]; check("12-month-mean benchmark recomputed with a plain loop (MAPE %)",round(float(np.mean(e2)),6),round(float(FB.mape_pct.iloc[1]),6),abs(np.mean(e2)-FB.mape_pct.iloc[1])<1e-9)
check("24 test months (2023 and 2024)",24,len(test),len(test)==24)
check("intervals contain the point estimate",3,int(((FB.mape_lo<=FB.mape_pct)&(FB.mape_pct<=FB.mape_hi)).sum()),((FB.mape_lo<=FB.mape_pct)&(FB.mape_pct<=FB.mape_hi)).all())

                 benchmark  test_months  mape_pct  mape_lo  mape_hi
      same month last year           24      8.40     6.41    10.57
mean of previous 12 months           24      6.59     5.00     8.19
        last month (naive)           24      6.84     4.91     8.87
lowest MAPE: mean of previous 12 months


## 8. Charts

In [9]:
o=rk.sort_values("base_rank")
fig,ax=plt.subplots(figsize=(8.5,4.6)); style(ax)
yy=np.arange(len(o))[::-1]
ax.hlines(yy,o.rank_lo,o.rank_hi,color=MUTED,lw=6,alpha=.8); ax.plot(o.base_rank,yy,"o",color=BLUE,ms=9,zorder=3)
ax.set_yticks(yy); ax.set_yticklabels(o.supplier_id,color=INK2); ax.set_xlim(.5,8.5); ax.set_xticks(range(1,9)); ax.set_xlabel("Risk rank (1 = riskiest)",color=INK2)
ax.set_title("Every supplier's rank interval spans most of the scale",loc="left",color=INK,fontsize=11); ax.yaxis.grid(False); ax.xaxis.grid(True,color=GRID,lw=.8)
fig.text(.01,-.02,"Dot = base rank; bar = 95% bootstrap interval, 5,000 resamples, seed 42.",fontsize=8.5,color=INK2); fig.tight_layout(); fig.savefig(CH/"w82_01_rank_intervals.png",dpi=150,facecolor=SURF,bbox_inches="tight"); plt.close(fig)
fig,ax=plt.subplots(figsize=(8.5,4.2)); style(ax)
bins=np.linspace(-1,1,41); ax.hist(null,bins=bins,color=MUTED,alpha=.85,label="shuffled supplier labels"); ax.hist(real,bins=bins,color=BLUE,alpha=.55,label="true supplier labels")
ax.axvline(real.mean(),color=BLUE,lw=1.5); ax.axvline(null.mean(),color=INK2,lw=1.5,ls="--"); ax.legend(frameon=False,labelcolor=INK2)
ax.set_xlabel("Spearman agreement between the rankings from two random halves of the POs",color=INK2); ax.set_title("The ranking does not repeat across random halves of the data",loc="left",color=INK,fontsize=11)
fig.tight_layout(); fig.savefig(CH/"w82_02_split_half.png",dpi=150,facecolor=SURF); plt.close(fig)
fig,ax=plt.subplots(figsize=(6.2,5.2)); ax.set_facecolor(SURF); fig.set_facecolor(SURF)
order=list(rk.supplier_id); Pm=PM.loc[order,order].values; im=ax.imshow(Pm,cmap="Blues",vmin=0,vmax=1)
ax.set_xticks(range(k)); ax.set_xticklabels(order,rotation=45,ha="right",color=INK2,fontsize=8); ax.set_yticks(range(k)); ax.set_yticklabels(order,color=INK2,fontsize=8)
for i in range(k):
    for j in range(k):
        if i!=j: ax.text(j,i,f"{Pm[i,j]:.2f}",ha="center",va="center",fontsize=7,color=INK)
ax.set_title("Share of resamples where the row supplier is riskier than the column",loc="left",color=INK,fontsize=9.5)
fig.tight_layout(); fig.savefig(CH/"w82_03_pairwise.png",dpi=150,facecolor=SURF); plt.close(fig)
check("three charts written",3,len(list(CH.glob("w82_0*.png"))),len(list(CH.glob("w82_0*.png")))==3)

## 9. Validation

In [10]:
check("every output table written",9,len([f for f in OUT.glob("w82_*.csv")]),len([f for f in OUT.glob("w82_*.csv")])==9)
check("money-bearing tables carry the procurement-scale label",3,sum("label" in pd.read_csv(OUT/f).columns for f in ["w82_risk_scorecard.csv","w82_rank_stability.csv","w82_exposure_context.csv"]),all("label" in pd.read_csv(OUT/f).columns for f in ["w82_risk_scorecard.csv","w82_rank_stability.csv","w82_exposure_context.csv"]))
V=pd.DataFrame(checks); V.insert(0,"Check_ID",range(1,len(V)+1)); V.to_csv("validation_report_week08_w82.csv",index=False)
assert (V.Status=="PASS").all(); print(len(V),"checks,",int((V.Status=="PASS").sum()),"pass")

38 checks, 38 pass
